In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Milestone 5**

**Milestone Assignment Objective: The Messy Mashup Challenge**
Your ultimate goal for this project is to build an end-to-end deep learning pipeline capable of classifying music genres in highly noisy environments. You will not just be writing snippets of code; you will be developing a complete system.

The questions below serve as your architectural blueprint. By answering them and verifying your outputs, you will incrementally construct a bug-free data generation process, a robust PyTorch DataLoader, and a state-of-the-art Audio Spectrogram Transformer (AST) model.

**Your Final Deliverables:**

**Build & Verify:** Use the questions below to construct and test your pipeline components.

**Train:** Once your architecture is verified, run your training loop! Let your model learn to separate the instrument stems from the environmental noise.

**Predict & Submit:** After training, you must write an inference script to process the unseen, unlabelled audio files listed in the test.csv file. Generate your predictions, format them into a submission.csv file, and upload your submission to the Kaggle competition leaderboard to report your final performance!

Treat these questions as milestones. If your code outputs the correct answers here, you can be confident that your foundation is solid before you spend hours training on the GPU. Good luck!

In [11]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.utils.data import random_split
from sklearn.model_selection import train_test_split
from transformers import AutoFeatureExtractor
from transformers import ASTForAudioClassification


# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

**Q1): Dataset Splitting Strategy Task:** You are building a lazy-loading "recipe" generator for the 10 genres.

Instead of loading audio into RAM, you will create a list of dictionaries. Assume you create exactly 100 recipe dictionaries per genre. Pass this combined list of recipes into sklearn.model_selection.train_test_split using test_size=0.2, shuffle=True, and random_state=42.

Question: Exactly how many recipe items will be allocated to your validation set (val_recipes) list?

In [4]:
# 10 genres × 100 recipes each
genres = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]

recipes = []

# create dummy "recipe" dicts
for genre in genres:
    for i in range(100):
        recipes.append({"genre": genre, "id": f"{genre}_{i}"})

print("Total recipes:", len(recipes))  # 1000

# split
train_recipes, val_recipes = train_test_split(
    recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Validation size:", len(val_recipes))

Total recipes: 1000
Validation size: 200


**Q2) On-the-Fly Mixing Dimensions Task:** In your training loop, you will load audio stems on the fly. 

Assume you use librosa.load(sr=16000, duration=10) to load 4 instrument stems and 1 noise file. You pad or truncate all 5 arrays to be exactly 160,000 samples long. 

You sum the 4 stems together to create a mix array, and then add the noise array scaled by an intensity weight of 0.2. 


Question: Before this final mix array is passed to the feature extractor, what must its exact 1D NumPy shape be? 

In [5]:
# simulate 4 stems + noise
stems = [np.ones(160000) for _ in range(4)]
noise = np.ones(160000)

# mix stems
mix = sum(stems)

# add noise (scaled)
mix = mix + 0.2 * noise

print(mix.shape)

(160000,)


**Q3) Question 3: Hugging Face Feature Extractor Shape Task:**


Initialize Hugging Face's AutoFeatureExtractor using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

To test your pipeline, create a dummy audio mix of exactly 160,000 ones using 
mix = np.ones(160000). 
Pass this dummy array into the extractor with sampling_rate=16000 and return_tensors="pt". 
Finally, extract the input_values from the resulting dictionary and apply .squeeze(0) to remove the default batch dimension. 


Question: What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process). In this format: [a,b] or (a,b)

In [9]:
extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

# dummy audio
mix = np.ones(160000)

inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

spec = inputs["input_values"].squeeze(0)

print(spec.shape)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

torch.Size([1024, 128])


**Q4) Model Architecture Initialization Task:**
Initialize the ASTForAudioClassification.from_pretrained model using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

Because we are classifying 10 genres instead of the original AudioSet classes, you must explicitly pass num_labels=10 and ignore_mismatched_sizes=True to replace the classification head. 

**Question: Write a quick loop using**

**sum(p.numel() for p in model.parameters() if p.requires_grad)**

to calculate the number of trainable parameters in this newly configured model. What is the exact integer value?

In [12]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable params:", trainable_params)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable params: 86196490


**Q5) Inference Normalization Math Task: In your test inference loop, right before feature extraction, the audio is normalized to prevent clipping using this exact formula:**

y = y / (np.max(np.abs(y)) + 1e-9). 

To test this, create a NumPy array: 
y_test = np.array([-0.85, 0.40, 0.20, -0.10]). 

Apply the normalization formula to it. 

**Question: What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?**

In [13]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)

print("Normalized:", y_norm)
print("Value at index 0:", round(y_norm[0], 3))

Normalized: [-1.          0.47058823  0.23529412 -0.11764706]
Value at index 0: -1.0
